In [ ]:
from app.services.pipeline import coletar_dados

from time import perf_counter

username = "felipe.cruz"
password = "#Gladoscruz.9851"
analise = "compra por necessidade"

dfs = coletar_dados(username, password, analise)

In [ ]:
import pickle

with open("snapshot_dfs.pkl", "wb") as f:
    pickle.dump(dfs, f)

print("Snapshot salvo.")



In [1]:
import pickle

dfs = {}
with open("snapshot_dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [ ]:
dfs.keys()
print (dfs.get('apoio_compras').columns)

In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        # Remove o ponto e garante que a coluna seja tratada como string
        ordens[col] = pd.to_datetime(
            ordens[col].astype(str).str.replace(".", "", regex=False),
            format="%d%m%Y",
            errors="coerce",
        )
    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))

    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    print (consumo.columns)
    consumo = consumo[
        [
            "Item",
            "Baixa",
            "Consumo",
            "Local Prod.",
            "OP",
            "Familia"
        ]
    ]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    print("consumo")
    print(consumo.columns)
    print("\nordens")
    print(ordens.columns)

    ######## Arvore ##########

    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )

    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )
    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

dict_keys(['apoio_compras', 'cons', 'ordens'])

consumo
Index(['Item', 'Baixa', 'Consumo', 'Local Prod.', 'Ordem Cons', 'Familia'], dtype='str')

ordens
Index(['Cliente', 'Fábrica', 'Ordem Prod', 'Pedido', 'Item', 'Saldo Prod',
       'Representante', 'Entrega Pedido', 'Data Abertura'],
      dtype='str')
